
# Retina-SEM: Superpixel Graph-Cut Vessel Segmentation (Final Notebook)

This notebook contains a full, **runnable** pipeline for retinal vessel segmentation using **SLIC superpixels + RAG + PyMaxflow graph cut**, with:
- Robust image loading (avoids dtype/object errors)
- Green-channel + CLAHE preprocessing
- Frangi vesselness (normalized to [0,1])
- Superpixel probabilities and **−log** unary costs
- Seed **hard pinning** (auto-seeded from vesselness percentiles if none provided)
- PyMaxflow min-cut on a region adjacency graph (RAG) built from mean-color similarity
- **Compatibility wrapper** (`run_one_compat`) that returns the legacy 8-tuple so existing
  code like `show_demo(...)` can be used without changes.


In [7]:

# === Retina-SEM Graph Cut Utilities (Deep Research Integration + Robust IO) ===
import numpy as np
import os
import cv2
from PIL import Image
from skimage import io, segmentation, graph, util
from skimage.filters import frangi
import maxflow

def _load_image(image_or_path):
    """Load an image as RGB uint8 robustly (path, PIL.Image, ndarray)."""
    # Path-like
    if isinstance(image_or_path, (str, bytes, os.PathLike)):
        data = np.fromfile(image_or_path, dtype=np.uint8)
        img_bgr = cv2.imdecode(data, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise ValueError(f"Failed to read image: {image_or_path}")
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    elif isinstance(image_or_path, Image.Image):
        img = np.array(image_or_path)
    else:
        img = np.asarray(image_or_path)

    # Normalize dtype to uint8
    if img.dtype == object:
        img = Image.fromarray(np.asarray(img))
        img = np.array(img)

    if img.ndim == 2:
        pass
    elif img.ndim == 3 and img.shape[2] >= 3:
        img = img[..., :3]  # RGB
    else:
        raise ValueError(f"Unsupported image shape: {img.shape}")

    if img.dtype == np.bool_:
        img = (img.astype(np.uint8) * 255)
    elif np.issubdtype(img.dtype, np.floating):
        img = np.clip(img, 0, 1)
        img = (img * 255).round().astype(np.uint8)
    elif img.dtype != np.uint8:
        maxv = np.iinfo(img.dtype).max if np.issubdtype(img.dtype, np.integer) else 255.0
        img = np.clip(img.astype(np.float32), 0, float(maxv))
        img = (img * (255.0 / maxv)).round().astype(np.uint8)
    return img

def _to_gray_green(img):
    """Return green channel (if RGB) or grayscale as uint8 (safe for odd dtypes)."""
    if img.ndim == 3 and img.shape[2] >= 2:
        g = img[..., 1]
    else:
        g = img

    if g.dtype == np.bool_:
        return (g.astype(np.uint8) * 255)
    elif np.issubdtype(g.dtype, np.floating):
        g = np.clip(g, 0, 1)
        return (g * 255).round().astype(np.uint8)
    elif g.dtype != np.uint8:
        maxv = np.iinfo(g.dtype).max if np.issubdtype(g.dtype, np.integer) else 255.0
        g = np.clip(g.astype(np.float32), 0, float(maxv))
        return (g * (255.0 / maxv)).round().astype(np.uint8)
    else:
        return g

def _clahe_u8(gray_u8, clip=2.0, tile=(8,8)):
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=tile)
    return clahe.apply(gray_u8)

def _compute_vesselness(gray_u8, sigmas=(1,2,3), beta=0.5, gamma=15.0):
    # skimage.frangi expects float image in [0,1]
    g = gray_u8.astype(np.float32) / 255.0
    vn = frangi(g, beta=beta, gamma=gamma, sigmas=sigmas, mode='nearest')
    vn = np.nan_to_num(vn, nan=0.0, posinf=0.0, neginf=0.0)
    # Normalize to [0,1]
    vmin, vmax = float(vn.min()), float(vn.max())
    if vmax > vmin:
        vn = (vn - vmin) / (vmax - vmin + 1e-6)
    else:
        vn = np.zeros_like(vn, dtype=np.float32)
    return vn.astype(np.float32)

def _build_rag_simple(img_rgb, labels):
    """Build a RAG using mean color distance; convert to similarity weights."""
    if img_rgb.ndim == 2:
        img_rgb = np.dstack([img_rgb]*3)
    rag = graph.rag_mean_color(img_rgb, labels, mode='distance')
    for u, v, data in rag.edges(data=True):
        dist = data.get('weight', 1.0)
        sim = 1.0 / (1.0 + float(dist))  # higher = more similar
        data['weight'] = sim
    return rag

def _spx_probs_from_vn(vn, labels):
    """Mean vesselness per superpixel (labels >= 0)."""
    num = int(labels.max()) + 1 if labels.size else 0
    p_spx = np.zeros(num, dtype=np.float32)
    cnt = np.zeros(num, dtype=np.int64)
    valid = labels >= 0
    ys, xs = np.nonzero(valid)
    for y, x in zip(ys, xs):
        lbl = labels[y, x]
        p_spx[lbl] += vn[y, x]
        cnt[lbl] += 1
    nz = cnt > 0
    p_spx[nz] /= cnt[nz]
    return p_spx


In [8]:

def graph_cut_segment(labels, rag, fg_ids, bg_ids, U_fg, U_bg, lam_unary=8, beta_pair=60):
    import numpy as np, maxflow
    num_nodes = int(labels.max()) + 1 if labels.size else 0

    # Scale unaries
    U_fg_scaled = lam_unary * U_fg.astype(np.float64)
    U_bg_scaled = lam_unary * U_bg.astype(np.float64)

    g = maxflow.Graph[float](num_nodes, max(1, num_nodes*6))
    nodes = g.add_nodes(num_nodes)

    # Terminal edges
    for i in range(num_nodes):
        g.add_tedge(nodes[i], float(U_fg_scaled[i]), float(U_bg_scaled[i]))

    # Pairwise edges
    for u, v, data in rag.edges(data=True):
        if u < 0 or v < 0:
            continue
        w = float(data.get('weight', 1.0))
        cap = beta_pair * w
        g.add_edge(nodes[u], nodes[v], cap, cap)

    g.maxflow()

    # Build mask
    mask = np.zeros(labels.shape, dtype=np.uint8)
    if labels.size:
        maxlbl = int(labels.max())
        for lbl in range(maxlbl + 1):
            if g.get_segment(nodes[lbl]) == 0:
                mask[labels == lbl] = 1

    # Debug summary
    fov = (labels >= 0)
    fg_ratio = float(mask[fov].mean()) if fov.any() else 0.0
    print(f"[graph_cut_segment] nodes={num_nodes}, FG={len(fg_ids)}, BG={len(bg_ids)}, "
          f"Ufg_mean={U_fg.mean():.4f}, Ubg_mean={U_bg.mean():.4f}, FG_ratio={fg_ratio:.3f}")
    return mask


In [9]:

def run_one(image_or_path, n_segments=600, compactness=10.0,
            lam_unary=8, beta_pair=60,
            fg_px=None, bg_px=None,
            frangi_sigmas=(1,2,3),
            clahe_clip=2.0, clahe_tile=(8,8)):
    """End-to-end vessel segmentation via superpixel graph-cut.
    Returns: mask (H,W) uint8 in {0,1}, aux dict with vn, labels, gray, rgb.
    """
    img = _load_image(image_or_path)
    rgb = img if img.ndim == 3 else np.dstack([img]*3)

    gray_u8 = _to_gray_green(rgb)
    gray_eq = _clahe_u8(gray_u8, clip=clahe_clip, tile=clahe_tile)
    vn = _compute_vesselness(gray_eq, sigmas=frangi_sigmas)

    labels = segmentation.slic(rgb, n_segments=n_segments, compactness=compactness, start_label=0)
    rag = _build_rag_simple(rgb, labels)

    # Seeds (auto if not provided)
    if fg_px is None or bg_px is None:
        hi, lo = 97.5, 10.0
        t_hi = np.percentile(vn, hi)
        t_lo = np.percentile(vn, lo)
        fg_px_auto = vn >= t_hi
        bg_px_auto = vn <= t_lo
        if fg_px is None: fg_px = fg_px_auto
        if bg_px is None: bg_px = bg_px_auto

    fg_ids = set(np.unique(labels[(fg_px) & (labels>=0)]).tolist())
    bg_ids = set(np.unique(labels[(bg_px) & (labels>=0)]).tolist())
    overlap = fg_ids & bg_ids
    if overlap:
        if len(fg_ids) <= len(bg_ids):
            fg_ids -= overlap
        else:
            bg_ids -= overlap

    # Unary costs from superpixel probabilities
    p_spx = _spx_probs_from_vn(vn, labels)
    eps = 1e-6
    U_fg = -np.log(p_spx + eps).astype(np.float64)
    U_bg = -np.log(1.0 - p_spx + eps).astype(np.float64)

    # Hard pin seeds
    BIG = 1e6
    if len(fg_ids):
        U_bg[list(fg_ids)] = BIG
    if len(bg_ids):
        U_fg[list(bg_ids)] = BIG

    mask = graph_cut_segment(labels, rag, fg_ids, bg_ids, U_fg, U_bg,
                             lam_unary=lam_unary, beta_pair=beta_pair)
    return mask.astype(np.uint8), {"vn": vn, "labels": labels, "gray": gray_eq, "rgb": rgb}


In [10]:

def run_one_compat(image_or_path, **kwargs):
    """Compatibility wrapper to match old signature:
    returns (gray, fov, vn, labels, fg_px, bg_px, seg, gt)"""
    mask, aux = run_one(image_or_path, **kwargs)
    vn = aux["vn"]
    labels = aux["labels"]
    gray = aux["gray"]
    fov = (labels >= 0).astype(np.uint8)

    hi, lo = 97.5, 10.0
    t_hi = np.percentile(vn, hi)
    t_lo = np.percentile(vn, lo)
    fg_px = (vn >= t_hi)
    bg_px = (vn <= t_lo)

    seg = mask.astype(np.uint8)
    gt = None
    return gray, fov, vn, labels, fg_px, bg_px, seg, gt

# Alias so legacy code like `show_demo(...)` keeps working:
run_one = run_one_compat
print("[INFO] Using compatibility alias: run_one -> run_one_compat (returns 8-tuple)." )


[INFO] Using compatibility alias: run_one -> run_one_compat (returns 8-tuple).


In [11]:

# Optional clean demo function; if you already have one, you can ignore this.
import matplotlib.pyplot as plt
from skimage.segmentation import mark_boundaries

def show_demo(p_path, lam_unary=8, beta_pair=60, n_segments=600):
    gray, fov, vn, labels, fg_px, bg_px, seg, gt = run_one(p_path, lam_unary=lam_unary, beta_pair=beta_pair, n_segments=n_segments)
    fig, axs = plt.subplots(2, 3, figsize=(12,8))
    axs[0,0].imshow(gray, cmap='gray'); axs[0,0].set_title('Green-CLAHE'); axs[0,0].axis('off')
    axs[0,1].imshow(vn, cmap='gray'); axs[0,1].set_title('Frangi vesselness [0,1]'); axs[0,1].axis('off')
    axs[0,2].imshow(mark_boundaries(np.dstack([gray]*3)/255.0, np.maximum(labels,0), color=(1,0,0)))
    axs[0,2].set_title('SLIC boundaries'); axs[0,2].axis('off')
    axs[1,0].imshow(fg_px, cmap='gray'); axs[1,0].set_title('FG seeds (auto)'); axs[1,0].axis('off')
    axs[1,1].imshow(bg_px, cmap='gray'); axs[1,1].set_title('BG seeds (auto)'); axs[1,1].axis('off')
    axs[1,2].imshow(seg, cmap='gray'); axs[1,2].set_title(f'Graph-Cut mask (λ={lam_unary}, β={beta_pair})'); axs[1,2].axis('off')
    plt.tight_layout(); plt.show()


In [14]:

# === Parameter Sweep (set test images and run) ===
test_images = []  # e.g., ["532_N.png", "example2.png"]
if not test_images:
    print("Set `test_images = [...]` to run the sweep.")
else:
    import csv
    lam_values = [6, 8, 10, 12]
    beta_values = [40, 60, 80, 100, 120]
    rows = [("image","lam_unary","beta_pair","fg_ratio")]
    for img_path in test_images:
        for lam in lam_values:
            for beta in beta_values:
                gray, fov, vn, labels, fg_px, bg_px, seg, gt = run_one(img_path, lam_unary=lam, beta_pair=beta)
                fg_ratio = float(seg.mean())
                flag = "" if 0.03 <= fg_ratio <= 0.15 else " <-- out of range"
                print(f"{img_path} | λ={lam}, β={beta}: FG_ratio={fg_ratio*100:.2f}%{flag}")
                # Save artifacts
                base = os.path.splitext(os.path.basename(img_path))[0]
                mask_path = f"{base}_mask_lam{lam}_beta{beta}.png"
                ov_path   = f"{base}_overlay_lam{lam}_beta{beta}.png"
                # Save mask
                cv2.imwrite(mask_path, (seg*255).astype(np.uint8))
                # Overlay in red
                rgb = np.dstack([gray]*3)
                rgb = rgb.copy()
                rgb[seg>0] = [255,0,0]
                cv2.imwrite(ov_path, cv2.cvtColor(rgb.astype(np.uint8), cv2.COLOR_RGB2BGR))
                rows.append((img_path, lam, beta, fg_ratio))
    with open("sweep_results.csv","w", newline="") as f:
        csv.writer(f).writerows(rows)
    print("Saved sweep_results.csv")


Set `test_images = [...]` to run the sweep.
